In [2]:
import os 
import pandas as pd 
import numpy as np
from dataclasses import dataclass
from pathlib import Path

In [3]:
pwd

'c:\\Users\\RSR\\PYTHON\\ServicenowGamification\\Research'

In [4]:
os.chdir('c:\\Users\\RSR\\PYTHON\\ServicenowGamification\\src')

In [5]:
@dataclass
class DataMetricsCalculationConfig:
    root_dir : Path 
    data_path : str 
    local_data_file : str

In [6]:
@dataclass(frozen=True)
class OracleConnectConfig: 
    username : str
    password : str
    host : str
    port : str
    service_name : str

In [7]:
from ServicenowGamification.Utils.Common import * 
from ServicenowGamification.Constant import * 
from ServicenowGamification.Config import *

from ServicenowGamification.Components.Data_Sns_Oracle_Loader import IncidentDataLoader

In [8]:
os.chdir('c:\\Users\\RSR\\PYTHON\\ServicenowGamification')

In [9]:
class ConfigurationManager:
    def __init__(self,
                 CONFIG_PATH=CONFIG_FILE_PATH,
                 PARAMS_PATH=PARAMS_FILE_PATH
                 ):
         print(CONFIG_PATH)
                 
         print(f"path config yaml {Path(CONFIG_PATH)}")
         self.config=Read_Yaml(Path(CONFIG_PATH))
         self.params=Read_Yaml(Path(PARAMS_PATH))
         self.Data_MetricsCalculation = self.config.Data_MetricsCalculation
         self.Oracle_Connect=self.config.Oracle_Connect
    def get_metricscalculationconfig(self)->DataMetricsCalculationConfig:
         Config=self.Data_MetricsCalculation
         create_directories([Config.root_dir])
         data_preprocessingconfig=DataMetricsCalculationConfig(
              root_dir=Config.root_dir,
              data_path=Config.data_path,
              local_data_file=Config.local_data_file
         )
         return data_preprocessingconfig
    def get_oracle_connect_config(self) -> OracleConnectConfig:
             Config=self.Oracle_Connect
             #create_directories([Config.root_dir])
             
             oracle_connect_config=OracleConnectConfig(
                 username=Config.username,
                 password=Config.password,           
                 host=Config.host,
                 port=Config.port,
                 service_name=Config.service_name
             )
             return oracle_connect_config  


In [ ]:
class MetricsCalculation:
    def __init__(self,config:DataMetricsCalculationConfig):
        self.config=config
        self.params = Read_Yaml(PARAMS_FILE_PATH)
        config_manager=ConfigurationManager()
        self.loader = IncidentDataLoader(config_manager.get_oracle_connect_config() )
    def MTTR(self):
        df=pd.read_csv(self.config.data_path)
        #print(f"\n*-------------- Calculate MTTR(Mean Time To Resolve) ----------------*\n")
        date_columns = ["opened_at", "resolved_at", "closed_at"]
        for col in date_columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")
        df["resolution_time"] = df["resolved_at"] - df["opened_at"]
        df["resolution_hours"] = ((df["resolved_at"] - df["opened_at"]).dt.total_seconds() / 3600)
        mttr = df["resolution_hours"].mean()
        print("MTTR:", mttr)
      
        local_path=self.config.local_data_file
        #print(f"Preprocessed data saved to: {local_path}")
        #df.to_csv(local_path, index=False) 

        # ============================================================
        # AGENT LEVEL MTTR
        # ============================================================

        # Make sure date columns are datetime
        df["opened_at"] = pd.to_datetime(df["opened_at"], errors="coerce")

        df["resolved_at"] = pd.to_datetime(df["resolved_at"],errors="coerce")

        df["closed_at"] = pd.to_datetime(df["closed_at"],errors="coerce" )


        # ============================================================
        # RESOLUTION HOURS
        # ============================================================

        df["resolution_hours"] = ( (df["resolved_at"] - df["opened_at"]).dt.total_seconds()/ 3600)


        # ============================================================
        # VALID INCIDENTS FOR MTTR
        # ============================================================

        mttr_df = df[ df["assigned_to"].notna()& df["opened_at"].notna()& df["resolved_at"].notna()].copy()


        # ============================================================
        # AGENT LEVEL METRICS
        # ============================================================

        agent_mttr = ( mttr_df
            .groupby("assigned_to")
            .agg(
                MTTR_hours=(
                    "resolution_hours",
                    "mean"
                ),

                Resolved_Incident_Count=(
                    "resolved_at",
                    "count"
                ),

                Closed_Incident_Count=(
                    "closed_at",
                    "count"
                )
            )
            .reset_index()
        )


        # ============================================================
        # TOTAL COMPLETED INCIDENTS
        # ============================================================

        agent_mttr["Resolved_or_Closed_Count"] = (
            agent_mttr["Resolved_Incident_Count"]
            + agent_mttr["Closed_Incident_Count"]
        )

        agent_mttr.rename(columns={"assigned_to": "Agent"},inplace=True)
        agent_mttr.rename(columns={"Resolved_or_Closed_Count": "Total"},inplace=True)
        
        # ============================================================
        # SAVE
        # ============================================================
        save_path = os.path.join(
                        self.config.root_dir,
                        "AgentLevelMTTR.csv"
                    )
        if self.params.Oracle['Loader'] is None:
         
            
            agent_mttr.to_csv(
                        save_path,
                        index=False
                    )
            
        elif self.params.Oracle['Loader']=='Oracle':
            print('InSide oracle ')
            agent_mttr.to_csv(
                save_path,
                index=False
                )
            self.loader.run(agent_mttr,'MTTR')

            agent_mttr.to_csv(
                save_path,
                index=False
            )

        print(
            "Agent Level MTTR saved to:",
            save_path
        )

        print(agent_mttr)

         



        return df    

    def calculate_art(self,df):
        """
        Average Resolution Time grouped by assignment group.
        """
        print("Art calculation")
        df=df

        # Only resolved incidents
        resolved_df = df[
            df["resolved_at"].notna()
            & df["assignment_group"].notna()
            & df["resolution_hours"].notna()
        ].copy()

        group_art = (
            resolved_df
            .groupby("assignment_group")
            .agg(
                Average_Resolution_Time=(
                    "resolution_hours",
                    "mean"
                ),
                Resolved_Incident_Count=(
                    "resolved_at",
                    "count"
                )
            )
            .reset_index()
        )

        group_art["Average_Resolution_Time"] = (
            group_art["Average_Resolution_Time"]
            .round(2)
        )

        group_art.rename(
            columns={
                "assignment_group": "Assignment_Group"
            },
            inplace=True
        )

        # ============================================================
        # SAVE
        # ============================================================
        save_path = os.path.join(self.config.root_dir,"GroupLevelArt.csv")
        if self.params.Oracle['Loader']   is None:
            group_art.to_csv(save_path,index=False)
            
        elif self.params.Oracle['Loader']=='Oracle':
            print("Inside oracle art group")
            group_art.to_csv(save_path,index=False)
            self.loader.run(group_art,'AgentGroupART')        
        

        #============================================================
        # Agent level ART
        # ============================================================


        agent_art = (resolved_df.groupby(["assignment_group", "assigned_to"])
                     .agg(Average_Resolution_Time=( "resolution_hours","mean"),
                          Resolved_Incident_Count=("resolved_at","count"))
                          .reset_index())

        # Round ART
        agent_art["Average_Resolution_Time"] = (
            agent_art["Average_Resolution_Time"]
            .round(2)
        )

        # Rename columns
        agent_art.rename(
            columns={
                "assignment_group": "Assignment_Group",
                "assigned_to": "Agent"
            },
            inplace=True
        )

        print("\nAgent Level ART")
        #print(agent_art)
    


        # save_path = os.path.join(
        #     self.config.root_dir,
        #     "AgentLevelArt.csv"
        #     )



        # ============================================================
        # SAVE
        # ============================================================
        save_path = os.path.join(self.config.root_dir,"GroupLevelArt.csv")
        if self.params.Oracle['Loader']   is None:
            agent_art.to_csv(save_path,index=False)
                    
        elif self.params.Oracle['Loader']=='Oracle':
            print("Inside oracle art agent")
            agent_art.to_csv(save_path,index=False)
            self.loader.run(agent_art,'AgentART')              

        return df,group_art


    def calculate_sla(self, df):
        df=df
        def get_status(row):
            priority = str(row["priority"])
                    #print(f"priority {priority}")
        
            if priority == "1 - Critical":
                return "Met" if row["resolution_hours"] <= self.params.sla.priority_1 else "Breached"
        
            elif priority == "2 - High":
                return "Met" if row["resolution_hours"] <= self.params.sla.priority_2 else "Breached"
        
            elif priority == "3 - Moderate":
                return "Met" if row["resolution_hours"] <= self.params.sla.priority_3 else "Breached"
        
            elif priority == "4 - Low":
                return "Met" if row["resolution_hours"] <= self.params.sla.priority_4 else "Breached"
        
            else:
                return "Breached"

        df["sla_status"] = df.apply(
            get_status,
            axis=1
        )

        compliance = round(

            (
                df["sla_status"]
                .eq("Met")
                .mean()
            ) * 100,

            2

        )

        print(
            f"\nOverall SLA Compliance : {compliance}%"
        )

        return df
    def calculate_fcr(self,df):

        df["fcr"] = np.where(

            (
                (df["reopen_count"] == 0)
                &
                (df["reassignment_count"] == 0)
            ),

            1,

            0

        )

        fcr = round(

            df["fcr"].mean() * 100,

            2

        )

        print(f"\nFirst Call Resolution : {fcr}%")

        return df
    def performance_metrics(self,df):
        df=df
        total = len(df)

        closed = len(

            df[
                df["state"] == "Closed"
            ]

        )

        open_incidents = len(

            df[
                df["state"] != "Closed"
            ]

        )

        sla = round(

            (
                df["sla_status"]
                .eq("Met")
                .mean()
            ) * 100,

            2

        )

        mttr = round(

            df["resolution_hours"]
            .mean(),

            2

        )

        fcr = round(

            df["fcr"]
            .mean() * 100,

            2

        )

        metrics = {
            "Total Incidents": total,
            "Closed Incidents": closed,
            "Open Incidents": open_incidents,
            "MTTR(Hours)": mttr,
            "SLA Compliance(%)": sla,
            "FCR(%)": fcr
        }

        print("\nPerformance Metrics")

        for key, value in metrics.items():

            print(f"{key} : {value}")

        return df,metrics
    


    

In [34]:
import sys

In [37]:
log_file = open("Log_DataMetricsCalculation.txt", "w")
sys.stdout = log_file
config = ConfigurationManager()

metrics_calculation_config = config.get_metricscalculationconfig()

metrics_calculation = MetricsCalculation(metrics_calculation_config)   # ✅ Correct
df=metrics_calculation.MTTR()
df,art=metrics_calculation.calculate_art(df)
# df=metrics_calculation.calculate_sla(df)
# df=metrics_calculation.calculate_fcr(df)
# df,metrics=metrics_calculation.performance_metrics(df)

sys.stdout = sys.__stdout__
log_file.close()